In [1]:
'''
task: classify syllogism validity with CGIF notation
models: gemma-2-2b-it, llama-3.2-3b-instruct, phi-3.5-mini-instruct
dataset: pfolio
evaluation: zero-shot
'''
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# start preparing for QA pipeline
! pip install -U accelerate
! pip install -U transformers
!pip install transformers
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 141.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00


In [3]:
import pandas as pd

pfolio_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/p-folio/data/pfolio_kr_gold_train.csv")

In [4]:
# evaluation metrics

import numpy as np
import re
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

def predict_answer(model, tokenizer, obj, subject, ref_relation=None, source_knowledge=None):
  # define notation grammar
  grammar = r"""
    start: program
    program: [stat]+
    stat: proposition newline | keyword* quantifier* symbol* leftparen* (quantifier symbol)* proposition rightparen* newline | keyword* (quantifier symbol)* leftparen* (quantifier symbol)* proposition rightparen* newline
    proposition: atomicproposition | complexproposition
    complexproposition: keyword* proposition keyword leftparen* (quantifier symbol)* proposition rightparen*
    atomicproposition: leftparen* term* leftparen* term* rightparen*
    !term: (LETTER+) (LETTER+|DIGIT+|"=" | "+" | "-" | "," | "≠" | "?")* | (DIGIT+) (LETTER+|DIGIT+|"=" | "+" | "-" | "," | "≠" | "?")*
    !leftparen: "[("
    !rightparen: ")]"
    !keyword: "~" | "?"
    !quantifier: "*" | "@every *"
    symbol: LETTER
    newline: /\n/

    %import common.LETTER
    %import common.DIGIT
    %import common.INT -> NUMBER
    %import common.ESCAPED_STRING -> STRING
    %import common.WS
    %ignore WS
"""
  # prepare prompt
  rag_prompt = f"""
  <start_of_turn>user
  You are an expert logician. You are given a syllogism in CGIF with premises between <PREMISES></PREMISES> and conclusion between <CONCLUSION></CONCLUSION> tags.
  The CGIF BNF grammar to understand and reason in the language is given in the <GRAMMAR></GRAMMAR> tags.
  <GRAMMAR>{grammar}</GRAMMAR>
  <PREMISES>{subject}</PREMISES>
  <CONCLUSION>{obj}</CONCLUSION>
  Classify the conclusion as "T" if true, "F" if false or "U" if uncertain based on the premises. Present your answer only between <output></output> tags.
  <end_of_turn>
  <start_of_turn>model
  """
  input_ids = tokenizer(rag_prompt, return_tensors="pt").to(model.device)
  response = model.generate(**input_ids, max_new_tokens=500)
  predicted_relation = tokenizer.decode(response[0])
  matches = re.findall('<output>(.*)</output>', predicted_relation, flags=re.DOTALL)
  res = re.findall(r"<output>(.*)", matches[-1])  # from ['</output> tags.\n  <end_of_turn>\n  <start_of_turn>model\n  <output>T'] to ['T']
  predicted_label = res[0] if res else "None" # take first element from list ['T'] to get 'T'

  print("*** Premises: \n", subject)
  print("*** Conclusion: \n", obj)
  print("*** True Label: \n", ref_relation)
  print("*** Predicted Label: \n", predicted_label)
  return predicted_label

In [5]:
def infer_from_ontology(dataset, model, tokenizer, mode='default', notation='NL'):
  evaluation_metrics_df = pd.DataFrame(columns=["Accuracy", "Precision", "Recall", "F1"])
  reference_labels = []
  predicted_labels = []
  for index, row in dataset.iterrows():
      conclusion = row["Conclusions - " + notation]
      premises = row["Premises - " + notation]
      label = row["Truth Values"]
      if mode.lower() == "grammar":
        # conduct query with RAG retrival of sources
        # set number of candidate answers to consider as half the total triple store axioms
        source_information = """BNF GRAMMAR"""
        print("*** RAG INFORMATION:", source_information)
      # predict answer with model
      predicted_label = predict_answer(model, tokenizer, conclusion, premises, label)
      reference_labels.append(label)
      predicted_labels.append(predicted_label)
  # fill evaluation metrics dataframe
  accuracy_metric = accuracy_score(reference_labels, predicted_labels)
  precision_metric = precision_score(reference_labels, predicted_labels, average="macro")
  recall_metric = recall_score(reference_labels, predicted_labels, average="macro")
  f1_metric = f1_score(reference_labels, predicted_labels, average="macro")
  evaluation_metrics_df["Accuracy"] = [accuracy_metric]
  evaluation_metrics_df["Precision"] = [precision_metric]
  evaluation_metrics_df["Recall"] = [recall_metric]
  evaluation_metrics_df["F1"] = [f1_metric]
  print("Classification Report:", classification_report(reference_labels, predicted_labels))
  print("*************** INFERENCE COMPLETE ***************")
  return reference_labels, predicted_labels, evaluation_metrics_df, accuracy_metric, precision_metric, recall_metric, f1_metric

In [6]:
import torch
import json
from tqdm import tqdm
import torch.nn as nn
from torch.optim import Adam
import nltk
import spacy
import string
import evaluate  # Bleu
from torch.utils.data import Dataset, DataLoader, RandomSampler
import pandas as pd
import numpy as np
import transformers
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

import warnings
warnings.filterwarnings("ignore")

In [7]:
# login to hugging face to have access to the model
!pip install huggingface_hub
from huggingface_hub import notebook_login
notebook_login()

In [9]:
# try rag search with gemma
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it")
# CPU Enabled uncomment below 👇🏽
#model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it")
# GPU Enabled use below 👇🏽
model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it", device_map="auto")

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [10]:
# experiment: ZS prediction without Grammar
ref_labels, pred_labels, eval_metrics_df, acc_metric, pr_metric, re_metric, f_metric = infer_from_ontology(pfolio_df, model, tokenizer, mode='default', notation='CGIF')

*** Premises: 
 [@every *x [(wildturkey[(?x)]  [(easternwildturkey[(?x)]  osceolawildturkey[(?x)]  gouldswildturkey[(?x)]  merriamswildturkey[(?x)]  riograndewildturkey[(?x)]  ocellatedwildturkey[(?x)])])]
~[(easternwildturkey[(tom)])]
~[(osceolawildturkey[(tom)])]
~[(gouldswildturkey[(tom)])]
~[(merriamswildturkey[(tom)]  riograndewildturkey[(tom)])]
wildturkey[(tom)]]
*** Conclusion: 
 [ocellatedwildturkey[(tom)]]
*** True Label: 
 T
*** Predicted Label: 
 T
*** Premises: 
 [@every *x [(wildturkey[(?x)]  [(easternwildturkey[(?x)]  osceolawildturkey[(?x)]  gouldswildturkey[(?x)]  merriamswildturkey[(?x)]  riograndewildturkey[(?x)]  ocellatedwildturkey[(?x)])])]
~[(easternwildturkey[(tom)])]
~[(osceolawildturkey[(tom)])]
~[(gouldswildturkey[(tom)])]
~[(merriamswildturkey[(tom)]  riograndewildturkey[(tom)])]
wildturkey[(tom)]]
*** Conclusion: 
 [easternwildturkey[(tom)]]
*** True Label: 
 F
*** Predicted Label: 
 T
*** Premises: 
 [@every *x [(wildturkey[(?x)]  [(easternwildturkey[(?x)]

In [11]:
# output results
print("***** ACCURACY *****")
print(acc_metric)
print("***** PRECISION *****")
print(pr_metric)
print("***** RECALL *****")
print(re_metric)
print("***** F1 *****")
print(f_metric)
eval_metrics_df

***** ACCURACY *****
0.48172757475083056
***** PRECISION *****
0.3922331422331422
***** RECALL *****
0.4528391541933951
***** F1 *****
0.37361097533157367


,Accuracy,Precision,Recall,F1
0,0.481728,0.392233,0.452839,0.373611


In [13]:
# try rag search with llama
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
# CPU Enabled uncomment below 👇🏽
#model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it")
# GPU Enabled use below 👇🏽
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", device_map="auto")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [14]:
# experiment: ZS prediction without Grammar
ref_labels, pred_labels, eval_metrics_df, acc_metric, pr_metric, re_metric, f_metric = infer_from_ontology(pfolio_df, model, tokenizer, mode='default', notation='CGIF')

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


*** Premises: 
 [@every *x [(wildturkey[(?x)]  [(easternwildturkey[(?x)]  osceolawildturkey[(?x)]  gouldswildturkey[(?x)]  merriamswildturkey[(?x)]  riograndewildturkey[(?x)]  ocellatedwildturkey[(?x)])])]
~[(easternwildturkey[(tom)])]
~[(osceolawildturkey[(tom)])]
~[(gouldswildturkey[(tom)])]
~[(merriamswildturkey[(tom)]  riograndewildturkey[(tom)])]
wildturkey[(tom)]]
*** Conclusion: 
 [ocellatedwildturkey[(tom)]]
*** True Label: 
 T
*** Predicted Label: 
 T


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


*** Premises: 
 [@every *x [(wildturkey[(?x)]  [(easternwildturkey[(?x)]  osceolawildturkey[(?x)]  gouldswildturkey[(?x)]  merriamswildturkey[(?x)]  riograndewildturkey[(?x)]  ocellatedwildturkey[(?x)])])]
~[(easternwildturkey[(tom)])]
~[(osceolawildturkey[(tom)])]
~[(gouldswildturkey[(tom)])]
~[(merriamswildturkey[(tom)]  riograndewildturkey[(tom)])]
wildturkey[(tom)]]
*** Conclusion: 
 [easternwildturkey[(tom)]]
*** True Label: 
 F
*** Predicted Label: 
 None


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

*** Premises: 
 [american[(maggiefriedman)]  screenwriter[(maggiefriedman)]  producer[(maggiefriedman)]
showrunnerof[(maggiefriedman  witchesofeastend)]  executiveproducerof[(maggiefriedman  witchesofeastend)]  lifetimetelevisionseries[(maggiefriedman)]
fantasydrama[(witchesofeastend)]  series[(witchesofeastend)]
produces[(maggiefriedman  eastwick)]  develops[(maggiefriedman  eastwick)]
series[(eastwick)]  airedon[(eastwick  abc)]]
*** Conclusion: 
 [develops[(maggiefriedman  witchesofeastend)]]
*** True Label: 
 U
*** Predicted Label: 
 None


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


*** Premises: 
 [largecomplex[(shafaq-asiman)]  largecomplex[(shafaq-asiman)]  offshore[(shafaq-asiman)]  geologicalstructures[(shafaq-asiman)]  in[(shafaq-asiman  caspiansea)]
northwestof[(baku  shafaq-asiman)]
@every *x @every *y [(northwestof[(?x  y)]  southeastof[(?y  x)])]]
*** Conclusion: 
 [southeastof[(baku  shafaq-asiman)]]
*** True Label: 
 U
*** Predicted Label: 
 None


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


*** Premises: 
 [largecomplex[(shafaq-asiman)]  largecomplex[(shafaq-asiman)]  offshore[(shafaq-asiman)]  geologicalstructures[(shafaq-asiman)]  in[(shafaq-asiman  caspiansea)]
northwestof[(baku  shafaq-asiman)]
@every *x @every *y [(northwestof[(?x  y)]  southeastof[(?y  x)])]]
*** Conclusion: 
 [*x [(largecomplex[(?x)]  southeastof[(?x  baku)])]]
*** True Label: 
 T
*** Predicted Label: 
 None
*** Premises: 
 [largecomplex[(shafaq-asiman)]  largecomplex[(shafaq-asiman)]  offshore[(shafaq-asiman)]  geologicalstructures[(shafaq-asiman)]  in[(shafaq-asiman  caspiansea)]
northwestof[(baku  shafaq-asiman)]
@every *x @every *y [(northwestof[(?x  y)]  southeastof[(?y  x)])]]
*** Conclusion: 
 [@every *x [(geologicalstructures[(?x)]  offshore[(?x)]  ~northwestof[(baku  x)])]]
*** True Label: 
 F
*** Predicted Label: 
 F
Classification Report:                                                 precision    recall  f1-score   support

                                                     0.00     

In [15]:
# output results
print("***** ACCURACY *****")
print(acc_metric)
print("***** PRECISION *****")
print(pr_metric)
print("***** RECALL *****")
print(re_metric)
print("***** F1 *****")
print(f_metric)
eval_metrics_df

***** ACCURACY *****
0.07641196013289037
***** PRECISION *****
0.1597222222222222
***** RECALL *****
0.027612367437741633
***** F1 *****
0.04511735765899656


,Accuracy,Precision,Recall,F1
0,0.076412,0.159722,0.027612,0.045117


In [17]:
# try rag search with phi
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct")
# CPU Enabled uncomment below 👇🏽
#model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it")
# GPU Enabled use below 👇🏽
model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3.5-mini-instruct", device_map="auto")

config.json: 0.00B [00:00, ?B/s]

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

In [18]:
# experiment: ZS prediction without Grammar
ref_labels, pred_labels, eval_metrics_df, acc_metric, pr_metric, re_metric, f_metric = infer_from_ontology(pfolio_df, model, tokenizer, mode='default', notation='CGIF')

*** Premises: 
 [@every *x [(wildturkey[(?x)]  [(easternwildturkey[(?x)]  osceolawildturkey[(?x)]  gouldswildturkey[(?x)]  merriamswildturkey[(?x)]  riograndewildturkey[(?x)]  ocellatedwildturkey[(?x)])])]
~[(easternwildturkey[(tom)])]
~[(osceolawildturkey[(tom)])]
~[(gouldswildturkey[(tom)])]
~[(merriamswildturkey[(tom)]  riograndewildturkey[(tom)])]
wildturkey[(tom)]]
*** Conclusion: 
 [ocellatedwildturkey[(tom)]]
*** True Label: 
 T
*** Predicted Label: 
 T</output>
*** Premises: 
 [@every *x [(wildturkey[(?x)]  [(easternwildturkey[(?x)]  osceolawildturkey[(?x)]  gouldswildturkey[(?x)]  merriamswildturkey[(?x)]  riograndewildturkey[(?x)]  ocellatedwildturkey[(?x)])])]
~[(easternwildturkey[(tom)])]
~[(osceolawildturkey[(tom)])]
~[(gouldswildturkey[(tom)])]
~[(merriamswildturkey[(tom)]  riograndewildturkey[(tom)])]
wildturkey[(tom)]]
*** Conclusion: 
 [easternwildturkey[(tom)]]
*** True Label: 
 F
*** Predicted Label: 
 T</output>
*** Premises: 
 [@every *x [(wildturkey[(?x)]  [(easte

In [19]:
# output results
print("***** ACCURACY *****")
print(acc_metric)
print("***** PRECISION *****")
print(pr_metric)
print("***** RECALL *****")
print(re_metric)
print("***** F1 *****")
print(f_metric)
eval_metrics_df

***** ACCURACY *****
0.132890365448505
***** PRECISION *****
0.1557504873294347
***** RECALL *****
0.03921994955494599
***** F1 *****
0.0590454288264148


,Accuracy,Precision,Recall,F1
0,0.13289,0.15575,0.03922,0.059045


In [20]:
# empty torch cuda cache
torch.cuda.empty_cache()

# delete model from cpu
del(model)